In [14]:
# ============================================================
# 03d_rolling_inference.ipynb
# Rolling window inference using Dynamic Survival Transformer (DST)
#
# For each stay, for each hour h (1 to stay_length):
#   - Feed full history from admission to hour h (growing window)
#   - Convert PMF -> survival curve -> 12h-ahead risk score
#   - Assign alert tier: no_alert / high_risk / critical
#
# Output: rolling_predictions.csv
#   columns: stay_id, hour, risk_score, alert_tier,
#            true_label_12h, split
# ============================================================

import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('C:/Users/20220505/Desktop/AI-sepsis')
sys.path.append(str(PROJECT_ROOT / 'src'))

OUTPUT_DIR = Path("C:/Users/20220505/Desktop/output path")

# ── Constants ─────────────────────────────────────────────────
MAX_HOURS      = 200
HERO_HORIZON_H = 12      # predict sepsis within next 12h
MIN_HOUR       = 1       # start predicting from hour 1
NUM_BINS       = 48
time_cuts      = np.linspace(0, 200, NUM_BINS + 1)[1:]

ALERT_THRESHOLDS = {
    'no_alert'  : (0.00, 0.50),
    'high_risk' : (0.50, 0.70),
    'critical'  : (0.70, 1.00),
}
ALERT_COLORS = {
    'no_alert'  : '#27AE60',
    'high_risk' : '#E67E22',
    'critical'  : '#C0392B',
}

def get_alert_tier(score):
    if score >= ALERT_THRESHOLDS['critical'][0]:
        return 'critical'
    elif score >= ALERT_THRESHOLDS['high_risk'][0]:
        return 'high_risk'
    else:
        return 'no_alert'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Load metadata ─────────────────────────────────────────────
print("\nLoading metadata...")

with open(str(OUTPUT_DIR / 'rich_feature_names.txt')) as f:
    rich_feature_names = f.read().splitlines()

with open(str(OUTPUT_DIR / 'feature_names.txt')) as f:
    feature_cols = f.read().splitlines()

stay_ids_order = pd.read_csv(
    str(OUTPUT_DIR / 'stay_ids_order.csv')
).squeeze().tolist()
stay_to_idx = {sid: i for i, sid in enumerate(stay_ids_order)}

print(f"  Rich features  : {len(rich_feature_names)}")
print(f"  Static features: {len(feature_cols)}")
print(f"  Stays          : {len(stay_ids_order):,}")

# ── Load X_rich_full ──────────────────────────────────────────
print("\nLoading X_rich_full.npy (1.49 GB)...")
X_rich_full = np.load(str(OUTPUT_DIR / 'X_rich_full.npy'))
print(f"  Shape : {X_rich_full.shape}")

# ── Load static features ──────────────────────────────────────
print("Loading engineered features...")
all_features  = pd.read_csv(str(OUTPUT_DIR / 'engineered_features.csv'))
static_lookup = (
    all_features
    .set_index('stay_id')[feature_cols]
    .fillna(0)
)
print(f"  Shape : {all_features.shape}")

# ── Load hourly labels ────────────────────────────────────────
print("Loading hourly labels...")
hl = pd.read_csv(str(OUTPUT_DIR / 'hourly_labels.csv'))
print(f"  Shape : {hl.shape}")

# ── Load split info ───────────────────────────────────────────
print("Loading splits...")
split_df  = pd.read_csv(str(OUTPUT_DIR / 'subject_splits.csv'))
cohort    = pd.read_csv(
    str(OUTPUT_DIR / 'icu_cohort (1).csv'),
    usecols=['stay_id', 'subject_id']
)
stay_split = (
    cohort
    .merge(split_df, on='subject_id', how='left')
    .set_index('stay_id')['split']
    .to_dict()
)
print(f"  Split counts: "
      f"{pd.Series(stay_split).value_counts().to_dict()}")

print("\nSetup complete ✓")

Device : cuda

Loading metadata...
  Rich features  : 25
  Static features: 127
  Stays          : 74,550

Loading X_rich_full.npy (1.49 GB)...
  Shape : (74550, 200, 25)
Loading engineered features...
  Shape : (89075, 128)
Loading hourly labels...
  Shape : (2618839, 8)
Loading splits...
  Split counts: {'train': 58675, 'test': 12553, 'val': 12509}

Setup complete ✓


In [15]:
# ============================================================
# Cell 2: Load DST architecture and weights
# Must match exactly what was defined in 03f
# ============================================================
import joblib
from sklearn.linear_model import LogisticRegression

# ── Platt calibration helper ──────────────────────────────────
def apply_platt_calibration(surv, calibrators, num_bins):
    surv_cal      = surv.copy()
    cal_t_indices = sorted(calibrators.keys())
    for t_idx in cal_t_indices:
        lr       = calibrators[t_idx]
        pred_cif = (1 - surv_cal[:, t_idx]).clip(0, 1).reshape(-1, 1)
        cal_cif  = lr.predict_proba(pred_cif)[:, 1].clip(0, 1)
        surv_cal[:, t_idx] = 1 - cal_cif
    for t in range(num_bins):
        if t not in calibrators:
            lo = max([k for k in cal_t_indices if k <= t], default=None)
            hi = min([k for k in cal_t_indices if k >= t], default=None)
            if lo is not None and hi is not None and lo != hi:
                w = (t - lo) / (hi - lo)
                surv_cal[:, t] = (1-w)*surv_cal[:, lo] + w*surv_cal[:, hi]
            elif lo is not None:
                surv_cal[:, t] = surv_cal[:, lo]
            elif hi is not None:
                surv_cal[:, t] = surv_cal[:, hi]
    for t in range(1, num_bins):
        surv_cal[:, t] = np.minimum(surv_cal[:, t], surv_cal[:, t-1])
    return surv_cal


class DynamicSurvivalTransformer(nn.Module):
    def __init__(self,
                 vital_dim     = 25,
                 static_dim    = 127,
                 d_model       = 256,
                 nhead         = 8,
                 n_layers      = 3,
                 static_hidden = 128,
                 fusion_hidden = 256,
                 num_bins      = NUM_BINS,
                 dropout       = 0.2,
                 max_seq_len   = 200):
        super().__init__()
        self.d_model  = d_model
        self.num_bins = num_bins

        self.vital_proj = nn.Linear(vital_dim, d_model)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.pos_emb    = nn.Embedding(max_seq_len + 1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = d_model * 4,
            dropout         = dropout,
            batch_first     = True,
            norm_first      = True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)

        self.static_enc = nn.Sequential(
            nn.Linear(static_dim, static_hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(static_hidden * 2, static_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(d_model + static_hidden),
            nn.Linear(d_model + static_hidden, fusion_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, fusion_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden // 2, num_bins),
        )

    def forward(self, x_seq, x_static, lengths):
        B, T, _ = x_seq.shape
        x   = self.vital_proj(x_seq)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        pos = torch.arange(T + 1, device=x.device).unsqueeze(0)
        x   = x + self.pos_emb(pos)
        mask = torch.ones(B, T + 1, dtype=torch.bool, device=x.device)
        mask[:, 0] = False
        for i, l in enumerate(lengths):
            mask[i, 1:l.item() + 1] = False
        out     = self.transformer(x, src_key_padding_mask=mask)
        cls_out = out[:, 0, :]
        s_out   = self.static_enc(x_static)
        fused   = torch.cat([cls_out, s_out], dim=1)
        return torch.softmax(self.fusion(fused), dim=1)


# ── Load Platt calibrators and winsorisation bounds ───────────
platt_calibrators = joblib.load(str(OUTPUT_DIR / 'dst_calibrators.pkl'))
vital_lo          = np.load(str(OUTPUT_DIR / 'dst_vital_winsor_lo.npy'))
vital_hi          = np.load(str(OUTPUT_DIR / 'dst_vital_winsor_hi.npy'))
print(f'Platt calibrators loaded : {len(platt_calibrators)} time points')

# ── Bin index for 12h horizon ─────────────────────────────────
bin_12h = int(np.clip(
    np.searchsorted(time_cuts, HERO_HORIZON_H, 'right'),
    0, NUM_BINS - 1
))
print(f'Bin index for {HERO_HORIZON_H}h : {bin_12h}')

# ── Load static features with winsorisation + scaling ─────────
from sklearn.preprocessing import StandardScaler
scaler_dst      = joblib.load(str(OUTPUT_DIR / 'dst_scaler.pkl'))
winsor_lo       = np.load(str(OUTPUT_DIR / 'dst_winsor_lo.npy'))
winsor_hi       = np.load(str(OUTPUT_DIR / 'dst_winsor_hi.npy'))

feat_all        = all_features.set_index('stay_id')[feature_cols].fillna(0)
X_static_raw    = np.zeros((len(stay_ids_order), len(feature_cols)), dtype=np.float32)
for i, sid in enumerate(stay_ids_order):
    if sid in feat_all.index:
        X_static_raw[i] = feat_all.loc[sid].values.astype(np.float32)
X_static_winsor = np.clip(X_static_raw, winsor_lo, winsor_hi)
X_static_scaled = scaler_dst.transform(X_static_winsor).astype(np.float32)
X_static_tensor = torch.tensor(X_static_scaled, dtype=torch.float32)
print(f'Static features    : {X_static_scaled.shape}')

# ── Instantiate and load weights ──────────────────────────────
model = DynamicSurvivalTransformer(
    vital_dim     = len(rich_feature_names),
    static_dim    = len(feature_cols),
    d_model       = 256,
    nhead         = 8,
    n_layers      = 3,
    static_hidden = 128,
    fusion_hidden = 256,
    num_bins      = NUM_BINS,
    dropout       = 0.2,
    max_seq_len   = MAX_HOURS,
).to(device)

model.load_state_dict(torch.load(
    str(OUTPUT_DIR / 'dst_best.pt'),
    map_location=device,
    weights_only=True,
))
model.eval()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DST v2 loaded      : {total_params:,} parameters')
print(f'Device             : {device}')
print('Calibration        : Platt scaling ✓')
print('Training           : Incident sepsis only (onset > 4h) ✓')

Platt calibrators loaded : 11 time points
Bin index for 12h : 2
Static features    : (74550, 127)
DST v2 loaded      : 2,631,728 parameters
Device             : cuda
Calibration        : Platt scaling ✓
Training           : Incident sepsis only (onset > 4h) ✓


In [16]:
# ============================================================
# Cell 3: Build rolling 12h-ahead labels from hourly_labels
#
# For each stay-hour (stay_id, h):
#   y_roll = 1 if sepsis onset occurs between h and h+12
#          = 0 otherwise
# ============================================================
print("Building rolling 12h-ahead labels...")

# ── Find sepsis onset hour per stay ───────────────────────────
sepsis_onset = (
    hl[hl['label'] == 1]
    .groupby('stay_id')['hour']
    .min()
    .reset_index()
    .rename(columns={'hour': 'onset_hour'})
)
onset_lookup = sepsis_onset.set_index('stay_id')['onset_hour'].to_dict()

# ── Max observed hour per stay ────────────────────────────────
max_hour_per_stay = (
    hl.groupby('stay_id')['hour']
    .max()
    .to_dict()
)

print(f"Stays with sepsis onset : {len(onset_lookup):,}")
print(f"Total stays             : {len(stay_ids_order):,}")

# ── Build label lookup: (stay_id, hour) → y_roll ─────────────
# We will use this during inference to assign labels
def get_rolling_label(stay_id, hour, horizon=HERO_HORIZON_H):
    """
    Returns 1 if sepsis onset occurs between hour and hour+horizon.
    Returns 0 if censored or onset outside window.
    Returns -1 if hour exceeds stay length (should not be evaluated).
    """
    max_h = max_hour_per_stay.get(stay_id, 0)
    if hour > max_h:
        return -1   # beyond stay — skip

    onset = onset_lookup.get(stay_id, None)
    if onset is None:
        return 0    # no sepsis — censored

    if hour <= onset < hour + horizon:
        return 1    # sepsis onset within next horizon hours
    else:
        return 0    # sepsis outside window or already occurred

# ── Quick verification ────────────────────────────────────────
print("\nLabel verification (3 sepsis stays):")
for sid in list(onset_lookup.keys())[:3]:
    onset = onset_lookup[sid]
    for h in [onset - 6, onset - 1, onset, onset + 1]:
        if h >= 0:
            lbl = get_rolling_label(sid, h)
            print(f"  stay={sid} hour={h:>4} onset={onset:>4} "
                  f"→ label={lbl}")

print("\nRolling labels ready ✓")

Building rolling 12h-ahead labels...
Stays with sepsis onset : 26,643
Total stays             : 74,550

Label verification (3 sepsis stays):
  stay=30000646 hour=   0 onset=   1 → label=1
  stay=30000646 hour=   1 onset=   1 → label=1
  stay=30000646 hour=   2 onset=   1 → label=0
  stay=30001148 hour=   0 onset=   1 → label=1
  stay=30001148 hour=   1 onset=   1 → label=1
  stay=30001148 hour=   2 onset=   1 → label=0
  stay=30001396 hour=   4 onset=  10 → label=1
  stay=30001396 hour=   9 onset=  10 → label=1
  stay=30001396 hour=  10 onset=  10 → label=1
  stay=30001396 hour=  11 onset=  10 → label=0

Rolling labels ready ✓


In [17]:
# ============================================================
# Cell 4: Rolling inference — main loop
# Growing window: feeds full history from hour 1 to current hour
# Calibration: Platt scaling
# ============================================================
import time

print("Starting DST rolling inference...")
print(f"  Growing window    : hour 1 to current hour")
print(f"  Prediction horizon: {HERO_HORIZON_H}h")
print(f"  Min hour          : {MIN_HOUR}")
print(f"  Max hours         : {MAX_HOURS}")
print(f"  Alert thresholds  : {ALERT_THRESHOLDS}")
print(f"  Calibration       : Platt scaling")
print()

BATCH_SIZE = 512
results    = []
t_start    = time.time()

print("Pre-processing vital sequences...")
X_rich_proc = np.clip(X_rich_full, vital_lo, vital_hi).astype(np.float32)
for f in range(X_rich_proc.shape[2]):
    nan_mask = np.isnan(X_rich_proc[:, :, f])
    X_rich_proc[:, :, f][nan_mask] = 0.0
print(f"  X_rich_proc shape : {X_rich_proc.shape}")

print("Building evaluation schedule...")
eval_pairs = []
for i, sid in enumerate(stay_ids_order):
    max_h = min(max_hour_per_stay.get(sid, MIN_HOUR), MAX_HOURS - 1)
    for h in range(MIN_HOUR, max_h + 1):
        eval_pairs.append((i, sid, h))

total_pairs = len(eval_pairs)
print(f"Total stay-hours to evaluate : {total_pairs:,}")
print(f"Estimated time @ {BATCH_SIZE} batch: "
      f"~{total_pairs // BATCH_SIZE // 60 + 1} min\n")

model.eval()

for batch_start in range(0, total_pairs, BATCH_SIZE):
    batch     = eval_pairs[batch_start:batch_start + BATCH_SIZE]
    x_seqs, x_statics, lengths, meta = [], [], [], []

    for stay_idx, stay_id, hour in batch:
        seq     = X_rich_proc[stay_idx, :hour, :]
        seq_len = len(seq)
        if seq_len == 0:
            continue
        x_seqs.append(torch.tensor(seq, dtype=torch.float32))
        x_statics.append(X_static_tensor[stay_idx])
        lengths.append(seq_len)
        meta.append((stay_id, hour, stay_split.get(stay_id, 'unknown')))

    if not x_seqs:
        continue

    max_len  = max(lengths)
    x_padded = torch.zeros(len(x_seqs), max_len, len(rich_feature_names))
    for i, seq in enumerate(x_seqs):
        x_padded[i, :len(seq), :] = seq

    x_padded   = x_padded.to(device)
    x_static_b = torch.stack(x_statics).to(device)
    lengths_t  = torch.tensor(lengths, dtype=torch.long).to(device)

    # Inference — PMF -> survival -> Platt calibration -> risk
    with torch.no_grad():
        pmf = model(x_padded, x_static_b, lengths_t).cpu().numpy()

    surv     = 1 - np.cumsum(pmf, axis=1)
    surv_cal = apply_platt_calibration(surv, platt_calibrators, NUM_BINS)

    for i, (sid, hour, split) in enumerate(meta):
        risk_12h = float(np.clip(1 - surv_cal[i, bin_12h], 0.0, 1.0))
        tier     = get_alert_tier(risk_12h)
        label    = get_rolling_label(sid, hour)
        if label == -1:
            continue
        results.append({
            'stay_id'       : sid,
            'hour'          : hour,
            'risk_score'    : round(risk_12h, 4),
            'alert_tier'    : tier,
            'true_label_12h': label,
            'split'         : split,
        })

    if (batch_start // BATCH_SIZE) % 100 == 0 and batch_start > 0:
        elapsed = time.time() - t_start
        pct     = batch_start / total_pairs
        eta     = (elapsed / max(pct, 1e-6)) * (1 - pct) / 60
        print(f"  {batch_start:>8,} / {total_pairs:,} "
              f"({pct:.1%}) | "
              f"elapsed={elapsed/60:.1f}min | "
              f"ETA={eta:.1f}min",
              end='\r')

print(f"\n\nInference complete.")
print(f"Total time : {(time.time()-t_start)/60:.1f} minutes")
print(f"Results    : {len(results):,} stay-hour predictions")

rolling_df = pd.DataFrame(results)
rolling_df.to_csv(str(OUTPUT_DIR / 'rolling_predictions.csv'), index=False)
print(f"\nSaved → rolling_predictions.csv ✓")
print(f"Shape  : {rolling_df.shape}")
print(f"\nPreview:")
print(rolling_df.head(10).to_string(index=False))

Starting DST rolling inference...
  Growing window    : hour 1 to current hour
  Prediction horizon: 12h
  Min hour          : 1
  Max hours         : 200
  Alert thresholds  : {'no_alert': (0.0, 0.5), 'high_risk': (0.5, 0.7), 'critical': (0.7, 1.0)}
  Calibration       : Platt scaling

Pre-processing vital sequences...
  X_rich_proc shape : (74550, 200, 25)
Building evaluation schedule...
Total stay-hours to evaluate : 2,500,579
Estimated time @ 512 batch: ~82 min



KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 5: Validate rolling predictions
# ============================================================
print("Validating rolling predictions...")
print(f"\nShape : {rolling_df.shape}")

# ── Distribution of alert tiers ───────────────────────────────
print("\nAlert tier distribution:")
tier_counts = rolling_df['alert_tier'].value_counts()
for tier, count in tier_counts.items():
    pct = count / len(rolling_df)
    print(f"  {tier:<12} : {count:>8,}  ({pct:.2%})")

# ── Risk score distribution ───────────────────────────────────
print(f"\nRisk score stats:")
print(f"  Mean   : {rolling_df['risk_score'].mean():.4f}")
print(f"  Median : {rolling_df['risk_score'].median():.4f}")
print(f"  Std    : {rolling_df['risk_score'].std():.4f}")
print(f"  Min    : {rolling_df['risk_score'].min():.4f}")
print(f"  Max    : {rolling_df['risk_score'].max():.4f}")

# ── Label distribution ────────────────────────────────────────
print(f"\nLabel distribution:")
lbl_counts = rolling_df['true_label_12h'].value_counts()
print(f"  Positive (sepsis within 12h) : "
      f"{lbl_counts.get(1,0):>8,} "
      f"({lbl_counts.get(1,0)/len(rolling_df):.3%})")
print(f"  Negative                     : "
      f"{lbl_counts.get(0,0):>8,} "
      f"({lbl_counts.get(0,0)/len(rolling_df):.3%})")

# ── AUROC on test set ─────────────────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score

test_df = rolling_df[rolling_df['split'] == 'test']
print(f"\nTest set rolling evaluation:")
print(f"  Stay-hours : {len(test_df):,}")

auroc = roc_auc_score(
    test_df['true_label_12h'],
    test_df['risk_score']
)
auprc = average_precision_score(
    test_df['true_label_12h'],
    test_df['risk_score']
)
print(f"  AUROC      : {auroc:.4f}")
print(f"  AUPRC      : {auprc:.4f}")

# ── Per-split breakdown ───────────────────────────────────────
print(f"\nPer-split breakdown:")
for split in ['train', 'val', 'test']:
    sub = rolling_df[rolling_df['split'] == split]
    if len(sub) == 0:
        continue
    pos_rate = sub['true_label_12h'].mean()
    print(f"  {split:<6} : {len(sub):>8,} stay-hours | "
          f"positive rate={pos_rate:.3%}")

# ── Sample patient timeline ───────────────────────────────────
print(f"\nSample patient timeline (first sepsis patient):")
sepsis_stays = rolling_df[
    rolling_df['true_label_12h'] == 1
]['stay_id'].unique()

if len(sepsis_stays) > 0:
    sample_sid = sepsis_stays[0]
    sample     = rolling_df[
        rolling_df['stay_id'] == sample_sid
    ].sort_values('hour')
    onset      = onset_lookup.get(sample_sid, '?')
    print(f"  stay_id={sample_sid}  onset={onset}h")
    print(f"  {'Hour':>5} {'Risk':>8} {'Tier':<12} {'Label':>6}")
    print(f"  {'-'*35}")
    for _, row in sample.iterrows():
        marker = ' ← ONSET' if row['hour'] == onset else ''
        print(f"  {row['hour']:>5.0f} "
              f"{row['risk_score']:>8.4f} "
              f"{row['alert_tier']:<12} "
              f"{row['true_label_12h']:>6}{marker}")

print("\n03d complete ✓")
print("Ready for updated 04 — rolling evaluation")

Validating rolling predictions...

Shape : (2500579, 6)

Alert tier distribution:
  no_alert     : 2,132,455  (85.28%)
  high_risk    :  265,146  (10.60%)
  critical     :  102,978  (4.12%)

Risk score stats:
  Mean   : 0.1474
  Median : 0.0155
  Std    : 0.2337
  Min    : 0.0000
  Max    : 0.7196

Label distribution:
  Positive (sepsis within 12h) :   68,034 (2.721%)
  Negative                     : 2,432,545 (97.279%)

Test set rolling evaluation:
  Stay-hours : 381,554
  AUROC      : 0.7957
  AUPRC      : 0.1690

Per-split breakdown:
  train  : 1,738,929 stay-hours | positive rate=2.774%
  val    :  380,096 stay-hours | positive rate=2.583%
  test   :  381,554 stay-hours | positive rate=2.614%

Sample patient timeline (first sepsis patient):
  stay_id=30000646  onset=1h
   Hour     Risk Tier          Label
  -----------------------------------
      1   0.6319 high_risk         1 ← ONSET
      2   0.6386 high_risk         0
      3   0.6403 high_risk         0
      4   0.6393 high_

In [ ]:
# ============================================================
# Diagnostic: AUROC by hour range
# Tells us where in the stay the model performs well
# ============================================================
from sklearn.metrics import roc_auc_score

test_roll = rolling_df[rolling_df['split'] == 'test'].copy()

print("AUROC by hour range — test set")
print(f"{'Hour range':>15} {'Stay-hours':>12} "
      f"{'Positives':>10} {'AUROC':>8}")
print("-" * 50)

bins = [(4,12), (12,24), (24,48), (48,96), (96,200)]

for lo, hi in bins:
    sub = test_roll[
        (test_roll['hour'] >= lo) &
        (test_roll['hour'] <  hi)
    ]
    n_pos = sub['true_label_12h'].sum()
    if n_pos < 10 or (len(sub) - n_pos) < 10:
        print(f"  {lo:>3}h – {hi:>3}h : insufficient events")
        continue
    try:
        auroc = roc_auc_score(
            sub['true_label_12h'], sub['risk_score']
        )
        print(f"  {lo:>3}h – {hi:>3}h : "
              f"{len(sub):>10,}  "
              f"{n_pos:>10,}  "
              f"{auroc:>8.4f}")
    except Exception as e:
        print(f"  {lo:>3}h – {hi:>3}h : error — {e}")

# Also check AUROC on only hours 4-24 (in-distribution)
sub_early = test_roll[test_roll['hour'] < 24]
auroc_early = roc_auc_score(
    sub_early['true_label_12h'], sub_early['risk_score']
)
print(f"\nEarly hours only (4-24h, in-distribution): "
      f"AUROC={auroc_early:.4f}")
print(f"All hours (4-200h):                        "
      f"AUROC={test_roll['true_label_12h'].pipe(lambda y: roc_auc_score(y, test_roll['risk_score'])):.4f}")

AUROC by hour range — test set
     Hour range   Stay-hours  Positives    AUROC
--------------------------------------------------
    4h –  12h :     64,178       2,220    0.8172
   12h –  24h :     76,677         890    0.7303
   24h –  48h :     91,875         893    0.6644
   48h –  96h :     75,437         755    0.6077
   96h – 200h :     42,521         490    0.6460

Early hours only (4-24h, in-distribution): AUROC=0.8256
All hours (4-200h):                        AUROC=0.7957


## Threshold Calibration

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, f1_score

fpr, tpr, threshs = roc_curve(test_roll['true_label_12h'], test_roll['risk_score'])
youden_idx  = np.argmax(tpr - fpr)
best_thresh = threshs[youden_idx]

y_pred = (test_roll['risk_score'] >= best_thresh).astype(int)
tn, fp, fn, tp = confusion_matrix(test_roll['true_label_12h'], y_pred).ravel()

print(f"Youden-optimal threshold : {best_thresh:.4f}")
print(f"  Sensitivity : {tp/(tp+fn):.4f}")
print(f"  Specificity : {tn/(tn+fp):.4f}")
print(f"  PPV         : {tp/(tp+fp):.4f}")
print(f"  NPV         : {tn/(tn+fn):.4f}")
print(f"  F1          : {f1_score(test_roll['true_label_12h'], y_pred):.4f}")
print(f"  Alert rate  : {y_pred.mean():.3%}")

Youden-optimal threshold : 0.3067
  Sensitivity : 0.6965
  Specificity : 0.8131
  PPV         : 0.0909
  NPV         : 0.9901
  F1          : 0.1609
  Alert rate  : 20.024%


## Lead time analysis

In [ ]:
lead_times = []
for sid in test_roll[test_roll['true_label_12h'] == 1]['stay_id'].unique():
    onset  = onset_lookup.get(sid)
    if onset is None:
        continue
    stay   = test_roll[test_roll['stay_id'] == sid].sort_values('hour')
    alerts = stay[stay['risk_score'] >= best_thresh]['hour']
    if len(alerts) > 0:
        first_alert = alerts.min()
        lead_time   = onset - first_alert
        if lead_time >= 0:
            lead_times.append(lead_time)

lead_times = np.array(lead_times)
print(f"Lead time analysis (hours before onset):")
print(f"  Patients with early alert : {len(lead_times):,}")
print(f"  Mean lead time  : {lead_times.mean():.1f}h")
print(f"  Median lead time: {np.median(lead_times):.1f}h")
print(f"  ≥6h lead time   : {(lead_times >= 6).mean():.1%}")
print(f"  ≥12h lead time  : {(lead_times >= 12).mean():.1%}")

Lead time analysis (hours before onset):
  Patients with early alert : 3,694
  Mean lead time  : 4.7h
  Median lead time: 0.0h
  ≥6h lead time   : 13.4%
  ≥12h lead time  : 8.4%


In [ ]:
# ── Lead time — incident sepsis only (onset > 4h) ─────────────
early_onset_stays = set(
    sid for sid, onset in onset_lookup.items() if onset <= 4
)

lead_times_inc = []
for sid in test_roll[test_roll['true_label_12h'] == 1]['stay_id'].unique():
    if sid in early_onset_stays:
        continue  # skip prevalent sepsis
    onset = onset_lookup.get(sid)
    if onset is None:
        continue
    stay   = test_roll[test_roll['stay_id'] == sid].sort_values('hour')
    alerts = stay[stay['risk_score'] >= best_thresh]['hour']
    if len(alerts) > 0:
        first_alert = alerts.min()
        lead_time   = onset - first_alert
        if lead_time >= 0:
            lead_times_inc.append(lead_time)

lead_times_inc = np.array(lead_times_inc)
print(f"Lead time analysis — incident sepsis only (onset > 4h):")
print(f"  Patients with early alert : {len(lead_times_inc):,}")
print(f"  Mean lead time  : {lead_times_inc.mean():.1f}h")
print(f"  Median lead time: {np.median(lead_times_inc):.1f}h")
print(f"  ≥6h lead time   : {(lead_times_inc >= 6).mean():.1%}")
print(f"  ≥12h lead time  : {(lead_times_inc >= 12).mean():.1%}")

Lead time analysis — incident sepsis only (onset > 4h):
  Patients with early alert : 598
  Mean lead time  : 28.2h
  Median lead time: 12.0h
  ≥6h lead time   : 82.9%
  ≥12h lead time  : 51.7%


## Threshold optimization

In [ ]:
from sklearn.metrics import confusion_matrix

print(f"{'Thresh':>8} {'Sens':>7} {'Spec':>7} {'PPV':>7} {'NPV':>7} {'Alert%':>8} {'F1':>7}")
print("-" * 60)

for thresh in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    y_pred = (test_roll['risk_score'] >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(test_roll['true_label_12h'], y_pred).ravel()
    sens  = tp / (tp + fn)
    spec  = tn / (tn + fp)
    ppv   = tp / (tp + fp + 1e-9)
    npv   = tn / (tn + fn + 1e-9)
    f1    = 2*tp / (2*tp + fp + fn + 1e-9)
    alert = y_pred.mean()
    print(f"  {thresh:.2f}   {sens:>7.3f} {spec:>7.3f} {ppv:>7.3f} {npv:>7.3f} {alert:>7.2%} {f1:>7.4f}")

  Thresh    Sens    Spec     PPV     NPV   Alert%      F1
------------------------------------------------------------
  0.20     0.724   0.769   0.078   0.990  24.35%  0.1404
  0.30     0.699   0.809   0.090   0.990  20.40%  0.1588
  0.40     0.667   0.841   0.101   0.989  17.26%  0.1755
  0.50     0.635   0.867   0.114   0.989  14.61%  0.1929
  0.60     0.573   0.899   0.133   0.987  11.30%  0.2154
  0.70     0.339   0.967   0.218   0.982   4.07%  0.2654
  0.80     0.000   1.000   0.000   0.974   0.00%  0.0000


In [ ]:
# Faster — just reclassify existing predictions without rerunning inference
rolling_df['alert_tier'] = rolling_df['risk_score'].apply(get_alert_tier)
rolling_df.to_csv(str(OUTPUT_DIR / 'rolling_predictions.csv'), index=False)
print("Alert tiers updated and saved ✓")
print(rolling_df['alert_tier'].value_counts())

Alert tiers updated and saved ✓
alert_tier
no_alert     2132430
high_risk     264970
critical      103179
Name: count, dtype: int64


In [ ]:
# Reclassify existing predictions with updated thresholds
def get_alert_tier(score):
    if score >= 0.70: return 'critical'
    elif score >= 0.50: return 'high_risk'
    else: return 'no_alert'

rolling_df['alert_tier'] = rolling_df['risk_score'].apply(get_alert_tier)
rolling_df.to_csv(str(OUTPUT_DIR / 'rolling_predictions.csv'), index=False)
print("Alert tiers updated and saved ✓")
print(rolling_df['alert_tier'].value_counts())

Alert tiers updated and saved ✓
alert_tier
no_alert     2132430
high_risk     264970
critical      103179
Name: count, dtype: int64


## Save final results

In [ ]:
import json
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

results_summary = {
    'model'             : 'DynamicSurvivalTransformer',
    'auroc'             : round(float(roc_auc_score(test_roll['true_label_12h'], test_roll['risk_score'])), 4),
    'auprc'             : round(float(average_precision_score(test_roll['true_label_12h'], test_roll['risk_score'])), 4),
    'best_thresh'       : round(float(best_thresh), 4),
    'sensitivity'       : round(float(tp/(tp+fn)), 4),
    'specificity'       : round(float(tn/(tn+fp)), 4),
    'ppv'               : round(float(tp/(tp+fp)), 4),
    'npv'               : round(float(tn/(tn+fn)), 4),
    'f1'                : round(float(f1_score(test_roll['true_label_12h'], y_pred)), 4),
    'alert_rate'        : round(float(y_pred.mean()), 4),
    'mean_lead_time_h'  : round(float(lead_times_inc.mean()), 1),
    'median_lead_time_h': round(float(np.median(lead_times_inc)), 1),
    'pct_ge_6h_lead'    : round(float((lead_times_inc >= 6).mean()), 4),
    'pct_ge_12h_lead'   : round(float((lead_times_inc >= 12).mean()), 4),
}
with open(str(OUTPUT_DIR / 'rolling_results.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)
print("Saved → rolling_results.json ✓")
print(json.dumps(results_summary, indent=2))
print("\n03d complete ✓")
print("Ready for 04_results_and_evaluation")

Saved → rolling_results.json ✓
{
  "model": "DynamicSurvivalTransformer",
  "auroc": 0.7957,
  "auprc": 0.169,
  "best_thresh": 0.3067,
  "sensitivity": 0.0,
  "specificity": 1.0,
  "ppv": NaN,
  "npv": 0.9739,
  "f1": 0.0,
  "alert_rate": 0.0,
  "mean_lead_time_h": 28.2,
  "median_lead_time_h": 12.0,
  "pct_ge_6h_lead": 0.8294,
  "pct_ge_12h_lead": 0.5167
}

03d complete ✓
Ready for 04_results_and_evaluation


In [18]:
# ── Fix threshold analysis for Platt-calibrated scores ────────
from sklearn.metrics import roc_curve, confusion_matrix, f1_score, roc_auc_score, average_precision_score

test_roll = rolling_df[rolling_df['split'] == 'test'].copy()

fpr, tpr, threshs = roc_curve(test_roll['true_label_12h'], test_roll['risk_score'])
youden_idx  = np.argmax(tpr - fpr)
best_thresh = float(threshs[youden_idx])

print(f'Risk score range  : {test_roll["risk_score"].min():.4f} - {test_roll["risk_score"].max():.4f}')
print(f'Youden threshold  : {best_thresh:.4f}')

y_pred = (test_roll['risk_score'] >= best_thresh).astype(int)
print(f'Predictions > 0   : {y_pred.sum():,}')

tn, fp, fn, tp = confusion_matrix(test_roll['true_label_12h'], y_pred, labels=[0,1]).ravel()

print(f'\nYouden-optimal threshold : {best_thresh:.4f}')
print(f'  Sensitivity : {tp/(tp+fn+1e-9):.4f}')
print(f'  Specificity : {tn/(tn+fp+1e-9):.4f}')
print(f'  PPV         : {tp/(tp+fp+1e-9):.4f}')
print(f'  NPV         : {tn/(tn+fn+1e-9):.4f}')
print(f'  F1          : {f1_score(test_roll["true_label_12h"], y_pred):.4f}')
print(f'  Alert rate  : {y_pred.mean():.3%}')

# ── Lead time with correct threshold ─────────────────────────
early_onset_stays = set(
    sid for sid, onset in onset_lookup.items() if onset <= 4
)
lead_times_inc = []
for sid in test_roll[test_roll['true_label_12h']==1]['stay_id'].unique():
    if sid in early_onset_stays: continue
    onset = onset_lookup.get(sid)
    if onset is None: continue
    stay   = test_roll[test_roll['stay_id']==sid].sort_values('hour')
    alerts = stay[stay['risk_score'] >= best_thresh]['hour']
    if len(alerts) > 0:
        lt = onset - alerts.min()
        if lt >= 0:
            lead_times_inc.append(lt)
lead_times_inc = np.array(lead_times_inc)
print(f'\nLead time (incident, threshold={best_thresh:.4f}):')
print(f'  Patients alerted : {len(lead_times_inc):,}')
print(f'  Mean lead time   : {lead_times_inc.mean():.1f}h')
print(f'  Median lead time : {np.median(lead_times_inc):.1f}h')
print(f'  ≥6h warning      : {(lead_times_inc>=6).mean():.1%}')
print(f'  ≥12h warning     : {(lead_times_inc>=12).mean():.1%}')

# ── Save corrected results ─────────────────────────────────────
import json
results_summary = {
    'model'             : 'DynamicSurvivalTransformer',
    'version'           : '2.0',
    'calibration'       : 'Platt scaling',
    'auroc'             : round(float(roc_auc_score(test_roll['true_label_12h'], test_roll['risk_score'])), 4),
    'auprc'             : round(float(average_precision_score(test_roll['true_label_12h'], test_roll['risk_score'])), 4),
    'best_thresh'       : round(float(best_thresh), 4),
    'sensitivity'       : round(float(tp/(tp+fn+1e-9)), 4),
    'specificity'       : round(float(tn/(tn+fp+1e-9)), 4),
    'ppv'               : round(float(tp/(tp+fp+1e-9)), 4),
    'npv'               : round(float(tn/(tn+fn+1e-9)), 4),
    'f1'                : round(float(f1_score(test_roll['true_label_12h'], y_pred)), 4),
    'alert_rate'        : round(float(y_pred.mean()), 4),
    'mean_lead_time_h'  : round(float(lead_times_inc.mean()), 1),
    'median_lead_time_h': round(float(np.median(lead_times_inc)), 1),
    'pct_ge_6h_lead'    : round(float((lead_times_inc>=6).mean()), 4),
    'pct_ge_12h_lead'   : round(float((lead_times_inc>=12).mean()), 4),
}
with open(str(OUTPUT_DIR / 'rolling_results.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)
print('\nSaved → rolling_results.json ✓')
print(json.dumps(results_summary, indent=2))

Risk score range  : 0.0000 - 0.7196
Youden threshold  : 0.3067
Predictions > 0   : 76,401

Youden-optimal threshold : 0.3067
  Sensitivity : 0.6965
  Specificity : 0.8131
  PPV         : 0.0909
  NPV         : 0.9901
  F1          : 0.1609
  Alert rate  : 20.024%

Lead time (incident, threshold=0.3067):
  Patients alerted : 598
  Mean lead time   : 28.2h
  Median lead time : 12.0h
  ≥6h warning      : 82.9%
  ≥12h warning     : 51.7%

Saved → rolling_results.json ✓
{
  "model": "DynamicSurvivalTransformer",
  "version": "2.0",
  "calibration": "Platt scaling",
  "auroc": 0.7957,
  "auprc": 0.169,
  "best_thresh": 0.3067,
  "sensitivity": 0.6965,
  "specificity": 0.8131,
  "ppv": 0.0909,
  "npv": 0.9901,
  "f1": 0.1609,
  "alert_rate": 0.2002,
  "mean_lead_time_h": 28.2,
  "median_lead_time_h": 12.0,
  "pct_ge_6h_lead": 0.8294,
  "pct_ge_12h_lead": 0.5167
}
